In [202]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_excel('BASE_2_Clientes_Manga.xlsx')
df.head()

,tipo_cliente,importancia_manga_produto_final_1a10,importancia_preco,importancia_certificacoes,aceita_refugo_como_mp,volume_tipico_compra_ton_mes,frequencia_compra_mensal
0,B2B,5,Média,Baixa,Sim,299,3
1,B2B,10,Alta,Baixa,Sim,1326,4
2,B2B,7,NaN,Média,Sim,1661,4
3,B2B,8,Alta,Média,Sim,1195,3
4,B2B,9,Média,Média,Sim,1783,4


In [203]:
df.isnull().sum()

tipo_cliente                              0
importancia_manga_produto_final_1a10      0
importancia_preco                       630
importancia_certificacoes                 0
aceita_refugo_como_mp                     0
volume_tipico_compra_ton_mes              0
frequencia_compra_mensal                  0
dtype: int64

In [204]:
target_col = "importancia_preco"
df["miss_preco"] = df[target_col].isna().astype(int)

df["miss_preco"].value_counts(dropna=False)


miss_preco
0    6370
1     630
Name: count, dtype: int64

In [205]:
from scipy.stats import chi2_contingency

cat_cols = ["tipo_cliente", "importancia_certificacoes", "aceita_refugo_como_mp"]

results_cat = []
for col in cat_cols:
    tab = pd.crosstab(df[col], df["miss_preco"])
    chi2, p, dof, expected = chi2_contingency(tab)
    results_cat.append((col, p))

results_cat = pd.DataFrame(results_cat, columns=["coluna", "p_value"]).sort_values("p_value")
display(results_cat)


,coluna,p_value
1,importancia_certificacoes,6.116289e-12
2,aceita_refugo_como_mp,1.130500e-08
0,tipo_cliente,5.056570e-05


In [206]:
from scipy.stats import mannwhitneyu

num_cols = ["importancia_manga_produto_final_1a10", "volume_tipico_compra_ton_mes", "frequencia_compra_mensal"]

def cliff_delta(x, y):
    # efeito: -1 a 1 (0 = sem diferença). Bom pra amostras grandes.
    x = np.asarray(x); y = np.asarray(y)
    gt = sum(i > j for i in x for j in y)
    lt = sum(i < j for i in x for j in y)
    return (gt - lt) / (len(x) * len(y))

results_num = []
g0 = df[df["miss_preco"] == 0]
g1 = df[df["miss_preco"] == 1]

for col in num_cols:
    x = g0[col].dropna()
    y = g1[col].dropna()
    stat, p = mannwhitneyu(x, y, alternative="two-sided")
    d = cliff_delta(x.sample(min(len(x), 1500), random_state=0), y.sample(min(len(y), 1500), random_state=0))  # amostra pra não explodir custo
    results_num.append((col, float(p), float(d), float(x.median()), float(y.median())))

results_num = pd.DataFrame(results_num, columns=["coluna", "p_value", "cliffs_delta", "mediana_sem_falta", "mediana_com_falta"]).sort_values("p_value")
display(results_num)


,coluna,p_value,cliffs_delta,mediana_sem_falta,mediana_com_falta
1,volume_tipico_compra_ton_mes,1.972747e-136,-0.591702,700.0,1556.0
2,frequencia_compra_mensal,4.709278e-13,0.169723,3.0,3.0
0,importancia_manga_produto_final_1a10,7.239807e-02,0.038786,8.0,7.0


In [207]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
df["miss_preco"] = df["importancia_preco"].isna().astype(int)
features = [
    "tipo_cliente",
    "importancia_manga_produto_final_1a10",
    "importancia_certificacoes",
    "aceita_refugo_como_mp",
    "volume_tipico_compra_ton_mes",
    "frequencia_compra_mensal",
]
X = df[features].copy()
y = df["miss_preco"].astype(int)
X = pd.get_dummies(X, drop_first=True)
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
X = sm.add_constant(X).astype(float)
model = sm.Logit(y, X).fit(disp=False)
summary = model.summary2().tables[1].sort_values("P>|z|")
display(summary)

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
volume_tipico_compra_ton_mes,0.001751,0.000089,19.577561,2.402498e-85,0.001576,0.001927
const,-5.238751,0.344524,-15.205777,3.237658e-52,-5.914005,-4.563497
importancia_certificacoes_Média,0.480162,0.123918,3.874841,1.066945e-04,0.237287,0.723036
aceita_refugo_como_mp_Sim,0.274564,0.104892,2.617582,8.855520e-03,0.068979,0.480148
importancia_certificacoes_Baixa,0.186738,0.166155,1.123878,2.610648e-01,-0.138920,0.512397
importancia_manga_produto_final_1a10,0.030725,0.029574,1.038918,2.988430e-01,-0.027239,0.088689
tipo_cliente_B2C,-0.147446,0.182092,-0.809734,4.180928e-01,-0.504341,0.209448
frequencia_compra_mensal,0.001932,0.073375,0.026327,9.789965e-01,-0.141881,0.145745


In [208]:
target = "importancia_preco"
df_observed = df[df[target].notna()].copy()

In [209]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

In [210]:
target = "importancia_preco"

features = [
    "volume_tipico_compra_ton_mes",
    "importancia_certificacoes",
    "aceita_refugo_como_mp",
    "tipo_cliente",
    "frequencia_compra_mensal",
    "importancia_manga_produto_final_1a10"
]

cat_features = [
    "importancia_certificacoes",
    "aceita_refugo_como_mp",
    "tipo_cliente"
]


In [211]:
df_train = df[df[target].notna()].copy()
df_missing = df[df[target].isna()].copy()

X = df_train[features]
y = df_train[target]

In [212]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [213]:
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train, cat_features=cat_features)



In [214]:
y_pred = model.predict(X_test)

# CatBoost às vezes retorna shape (n, 1)
if isinstance(y_pred, (np.ndarray,)) and y_pred.ndim > 1:
    y_pred = y_pred.ravel()

# se vier como lista de listas, também resolve
y_pred = np.array(y_pred).ravel()

# =========================
# MÉTRICAS (ordem prática)
# =========================
f1_macro = f1_score(y_test, y_pred, average="macro")
recall_macro = recall_score(y_test, y_pred, average="macro")
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print("✅ Métricas (macro):")
print(f"1) F1 macro           : {f1_macro:.4f}")
print(f"2) Recall macro       : {recall_macro:.4f}")
print(f"3) Precision macro    : {precision_macro:.4f}")
print(f"4) Balanced accuracy  : {bal_acc:.4f}")

# =========================
# MATRIZ DE CONFUSÃO
# =========================
labels = sorted(pd.Series(y_test).dropna().unique().tolist())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"real_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])

print("\n✅ Matriz de confusão:")
display(cm_df)

# =========================
# (extra útil) relatório por classe
# =========================
print("\n✅ Relatório por classe:")
print(classification_report(y_test, y_pred, zero_division=0))

✅ Métricas (macro):
1) F1 macro           : 0.6562
2) Recall macro       : 0.6586
3) Precision macro    : 0.6555
4) Balanced accuracy  : 0.6586

✅ Matriz de confusão:


,pred_Alta,pred_Baixa,pred_Média
real_Alta,517,0,111
real_Baixa,4,168,67
real_Média,147,77,183



✅ Relatório por classe:
              precision    recall  f1-score   support

        Alta       0.77      0.82      0.80       628
       Baixa       0.69      0.70      0.69       239
       Média       0.51      0.45      0.48       407

    accuracy                           0.68      1274
   macro avg       0.66      0.66      0.66      1274
weighted avg       0.67      0.68      0.68      1274



In [215]:
from sklearn.metrics import f1_score

pred_train = model.predict(X_train).ravel()
pred_test  = model.predict(X_test).ravel()

print("F1 macro treino:", f1_score(y_train, pred_train, average="macro"))
print("F1 macro teste :", f1_score(y_test,  pred_test,  average="macro"))


F1 macro treino: 0.6751106413147324
F1 macro teste : 0.6562056274019658


In [216]:
target = "importancia_preco"

df_missing = df[df[target].isna()].copy()
pred_missing = model.predict(df_missing[features])

# garante formato correto
import numpy as np
pred_missing = np.array(pred_missing).ravel()

df.loc[df[target].isna(), target] = pred_missing


In [217]:
dist_before = (
    df_observed[target]
    .value_counts()
    .rename("antes")
)

dist_after = (
    df[target]
    .value_counts()
    .rename("depois")
)

dist_compare = pd.concat([dist_before, dist_after], axis=1)
display(dist_compare)


,antes,depois
importancia_preco,,
Alta,2969,3590
Média,2199,2208
Baixa,1202,1202


In [218]:
dist_before_pct = (
    df_observed[target]
    .value_counts(normalize=True)
    .rename("antes_%")
)

dist_after_pct = (
    df[target]
    .value_counts(normalize=True)
    .rename("depois_%")
)

dist_compare_pct = pd.concat([dist_before_pct, dist_after_pct], axis=1)
display(dist_compare_pct)


,antes_%,depois_%
importancia_preco,,
Alta,0.466091,0.512857
Média,0.345212,0.315429
Baixa,0.188697,0.171714


In [219]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    balanced_accuracy_score, confusion_matrix, classification_report,
    mean_absolute_error
)

# =========================
# CONFIG
# =========================
target = "importancia_preco"

features_reg = [
    "volume_tipico_compra_ton_mes",
    "importancia_certificacoes",
    "aceita_refugo_como_mp"
]

# mapeamentos (como você pediu)
map_cert = {"Baixa": 1, "Média": 2, "Alta": 3}
map_refugo = {"Não": 0, "Nao": 0, "Sim": 1}  # inclui variações comuns
map_target = {"Baixa": 1, "Média": 2, "Alta": 3}
inv_target = {1: "Baixa", 2: "Média", 3: "Alta"}

# =========================
# DATASET (somente linhas com target observado)
# =========================
df_obs = df[df[target].notna()].copy()

# features numéricas/ordinais
df_obs["importancia_certificacoes_num"] = df_obs["importancia_certificacoes"].map(map_cert)
df_obs["aceita_refugo_como_mp_num"] = df_obs["aceita_refugo_como_mp"].map(map_refugo)

# X / y
X = df_obs[["volume_tipico_compra_ton_mes", "importancia_certificacoes_num", "aceita_refugo_como_mp_num"]].copy()
y = df_obs[target].map(map_target).astype(float)

# segurança: remover linhas que viraram NaN por mapeamento
mask_ok = X.notna().all(axis=1) & y.notna()
X = X[mask_ok]
y = y[mask_ok]

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# =========================
# MODELO (REGRESSÃO)
# =========================
reg = CatBoostRegressor(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

reg.fit(X_train, y_train)

# =========================
# PREDIÇÃO CONTÍNUA -> ORDINAL (1,2,3) -> CLASSE
# =========================
y_pred_cont = reg.predict(X_test)

# converter para 1/2/3
y_pred_ord = np.rint(y_pred_cont)          # arredonda para o inteiro mais próximo
y_pred_ord = np.clip(y_pred_ord, 1, 3)     # garante dentro do range
y_pred_ord = y_pred_ord.astype(int)

# y_test já é numérico 1/2/3
y_test_ord = y_test.astype(int)

# converter para rótulos (para relatório/matriz)
y_pred_lbl = pd.Series(y_pred_ord).map(inv_target).values
y_test_lbl = pd.Series(y_test_ord).map(inv_target).values

# =========================
# MÉTRICAS (ORDINAL + CLASSIFICAÇÃO)
# =========================
mae_ord = mean_absolute_error(y_test_ord, y_pred_ord)

f1_macro = f1_score(y_test_lbl, y_pred_lbl, average="macro")
recall_macro = recall_score(y_test_lbl, y_pred_lbl, average="macro")
precision_macro = precision_score(y_test_lbl, y_pred_lbl, average="macro", zero_division=0)
bal_acc = balanced_accuracy_score(y_test_lbl, y_pred_lbl)

print("✅ CatBoostRegressor (ordinal) — Resultados")
print(f"MAE ordinal (1-3):      {mae_ord:.4f}")
print(f"F1 macro:              {f1_macro:.4f}")
print(f"Recall macro:          {recall_macro:.4f}")
print(f"Precision macro:       {precision_macro:.4f}")
print(f"Balanced accuracy:     {bal_acc:.4f}")

labels = ["Alta", "Baixa", "Média"]  # mesma ordem visual que você vinha usando
cm = confusion_matrix(y_test_lbl, y_pred_lbl, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"real_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])

print("\n✅ Matriz de confusão:")
display(cm_df)

print("\n✅ Relatório por classe:")
print(classification_report(y_test_lbl, y_pred_lbl, zero_division=0))

✅ CatBoostRegressor (ordinal) — Resultados
MAE ordinal (1-3):      0.3050
F1 macro:              0.6539
Recall macro:          0.6614
Precision macro:       0.6510
Balanced accuracy:     0.6614

✅ Matriz de confusão:


,pred_Alta,pred_Baixa,pred_Média
real_Alta,614,0,101
real_Baixa,0,168,74
real_Média,154,98,191



✅ Relatório por classe:
              precision    recall  f1-score   support

        Alta       0.80      0.86      0.83       715
       Baixa       0.63      0.69      0.66       242
       Média       0.52      0.43      0.47       443

    accuracy                           0.69      1400
   macro avg       0.65      0.66      0.65      1400
weighted avg       0.68      0.69      0.69      1400



In [220]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    f1_score,
    balanced_accuracy_score
)

# =========================
# FUNÇÃO AUXILIAR
# =========================
def evaluate_ordinal(model, X, y_true_num, inv_map):
    """
    model: CatBoostRegressor treinado
    X: features
    y_true_num: target numérico (1,2,3)
    inv_map: mapeamento num -> label
    """
    # predição contínua
    y_pred_cont = model.predict(X)

    # contínuo -> ordinal
    y_pred_ord = np.rint(y_pred_cont)
    y_pred_ord = np.clip(y_pred_ord, 1, 3).astype(int)

    # métricas ordinais
    mae = mean_absolute_error(y_true_num, y_pred_ord)

    # converter para rótulos
    y_true_lbl = pd.Series(y_true_num).map(inv_map)
    y_pred_lbl = pd.Series(y_pred_ord).map(inv_map)

    f1 = f1_score(y_true_lbl, y_pred_lbl, average="macro")
    bal_acc = balanced_accuracy_score(y_true_lbl, y_pred_lbl)

    return mae, f1, bal_acc


# =========================
# AVALIAÇÃO TREINO vs TESTE
# =========================
inv_target = {1: "Baixa", 2: "Média", 3: "Alta"}

mae_train, f1_train, bal_train = evaluate_ordinal(
    reg, X_train, y_train.astype(int), inv_target
)

mae_test, f1_test, bal_test = evaluate_ordinal(
    reg, X_test, y_test.astype(int), inv_target
)

# =========================
# RESULTADOS
# =========================
results = pd.DataFrame({
    "Treino": [mae_train, f1_train, bal_train],
    "Teste":  [mae_test,  f1_test,  bal_test],
    "Delta (Treino - Teste)": [
        mae_train - mae_test,
        f1_train - f1_test,
        bal_train - bal_test
    ]
}, index=[
    "MAE ordinal (↓ melhor)",
    "F1 macro (↑ melhor)",
    "Balanced accuracy (↑ melhor)"
])

display(results)


,Treino,Teste,Delta (Treino - Teste)
MAE ordinal (↓ melhor),0.288929,0.305000,-0.016071
F1 macro (↑ melhor),0.676953,0.653885,0.023067
Balanced accuracy (↑ melhor),0.673718,0.661369,0.012349


In [221]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    f1_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

# =========================
# CONFIG
# =========================
target = "importancia_preco"

# mapeamentos
map_cert = {"Baixa": 1, "Média": 2, "Alta": 3}
map_refugo = {"Não": 0, "Nao": 0, "Sim": 1}
map_target = {"Baixa": 1, "Média": 2, "Alta": 3}
inv_target = {1: "Baixa", 2: "Média", 3: "Alta"}

# =========================
# DATASET (somente observado)
# =========================
df_obs = df[df[target].notna()].copy()

df_obs["importancia_certificacoes_num"] = df_obs["importancia_certificacoes"].map(map_cert)
df_obs["aceita_refugo_como_mp_num"] = df_obs["aceita_refugo_como_mp"].map(map_refugo)

X = df_obs[[
    "volume_tipico_compra_ton_mes",
    "importancia_certificacoes_num",
    "aceita_refugo_como_mp_num"
]].copy()

y = df_obs[target].map(map_target).astype(float)

# remove linhas que viraram NaN por mapeamento
mask_ok = X.notna().all(axis=1) & y.notna()
X = X[mask_ok]
y = y[mask_ok]

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# =========================
# MODELO
# =========================
rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,        # pode ajustar depois
    min_samples_leaf=3,    # ajuda a reduzir overfitting
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# =========================
# FUNÇÃO: avaliar ordinal
# =========================
def eval_ordinal_regressor(model, X, y_true_num):
    y_pred_cont = model.predict(X)

    y_pred_ord = np.rint(y_pred_cont)
    y_pred_ord = np.clip(y_pred_ord, 1, 3).astype(int)
    y_true_ord = y_true_num.astype(int)

    mae = mean_absolute_error(y_true_ord, y_pred_ord)

    y_pred_lbl = pd.Series(y_pred_ord).map(inv_target).values
    y_true_lbl = pd.Series(y_true_ord).map(inv_target).values

    f1 = f1_score(y_true_lbl, y_pred_lbl, average="macro")
    recall = recall_score(y_true_lbl, y_pred_lbl, average="macro")
    precision = precision_score(y_true_lbl, y_pred_lbl, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true_lbl, y_pred_lbl)

    return mae, f1, recall, precision, bal_acc, y_true_lbl, y_pred_lbl


# =========================
# TESTE (métricas + matriz)
# =========================
mae, f1, recall, precision, bal_acc, y_true_lbl, y_pred_lbl = eval_ordinal_regressor(rf, X_test, y_test)

print("✅ RandomForestRegressor (ordinal) — Resultados (TESTE)")
print(f"MAE ordinal (1-3):      {mae:.4f}")
print(f"F1 macro:              {f1:.4f}")
print(f"Recall macro:          {recall:.4f}")
print(f"Precision macro:       {precision:.4f}")
print(f"Balanced accuracy:     {bal_acc:.4f}")

labels = ["Alta", "Baixa", "Média"]
cm = confusion_matrix(y_true_lbl, y_pred_lbl, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"real_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])

print("\n✅ Matriz de confusão:")
display(cm_df)

print("\n✅ Relatório por classe:")
print(classification_report(y_true_lbl, y_pred_lbl, zero_division=0))

# =========================
# OVERFITTING: treino vs teste
# =========================
mae_tr, f1_tr, rec_tr, prec_tr, bal_tr, _, _ = eval_ordinal_regressor(rf, X_train, y_train)
mae_te, f1_te, rec_te, prec_te, bal_te, _, _ = eval_ordinal_regressor(rf, X_test, y_test)

overfit_df = pd.DataFrame({
    "Treino": [mae_tr, f1_tr, bal_tr],
    "Teste":  [mae_te, f1_te, bal_te],
    "Delta (Treino - Teste)": [mae_tr - mae_te, f1_tr - f1_te, bal_tr - bal_te]
}, index=[
    "MAE ordinal (↓ melhor)",
    "F1 macro (↑ melhor)",
    "Balanced accuracy (↑ melhor)"
])

print("\n✅ Overfitting check:")
display(overfit_df)


✅ RandomForestRegressor (ordinal) — Resultados (TESTE)
MAE ordinal (1-3):      0.3150
F1 macro:              0.6482
Recall macro:          0.6503
Precision macro:       0.6480
Balanced accuracy:     0.6503

✅ Matriz de confusão:


,pred_Alta,pred_Baixa,pred_Média
real_Alta,604,0,111
real_Baixa,3,159,80
real_Média,155,89,199



✅ Relatório por classe:
              precision    recall  f1-score   support

        Alta       0.79      0.84      0.82       715
       Baixa       0.64      0.66      0.65       242
       Média       0.51      0.45      0.48       443

    accuracy                           0.69      1400
   macro avg       0.65      0.65      0.65      1400
weighted avg       0.68      0.69      0.68      1400


✅ Overfitting check:


,Treino,Teste,Delta (Treino - Teste)
MAE ordinal (↓ melhor),0.238750,0.315000,-0.076250
F1 macro (↑ melhor),0.733859,0.648215,0.085644
Balanced accuracy (↑ melhor),0.723825,0.650330,0.073495
